In [ ]:
%load_ext autoreload
%autoreload 2

# Backtest with Verbal RL

In [ ]:
import sys
from pathlib import Path

sys.path.append(
    Path.cwd().parents[1].as_posix()
)

In [ ]:
from uuid import uuid4

import google
import pandas as pd

from google.cloud import bigquery
from google.cloud import bigquery_storage_v1
from langchain import Wikipedia
from langchain_core.documents import Document
from langchain.agents.react.base import DocstoreExplorer
from langchain_community.docstore.in_memory import InMemoryDocstore
from vertexai.generative_models import GenerativeModel, GenerationConfig

# Defines

## GCP

In [ ]:
_, PROJECT_ID = google.auth.default()
REGION = "europe-west4"
BUCKET_NAME = "fin-news"

# VertexAI Params
model_id = "gemini-2.5-flash"

In [ ]:
# BigQuery client
bigquery_client = bigquery.Client(
    project=PROJECT_ID,
    location=REGION
)

bqstorage = bigquery_storage_v1.BigQueryReadClient()

## Constants

In [ ]:
train_start = '2018-01-01'
train_end = '2023-01-01'

test_end = '2023-12-01'

# How far back are we looking
context_lookback = '60D'
holding_period = '30D'
train_stride = '10D'

## Data - Prices

**TODO** I need to calcualte alphas (difference in % returns vs the index) - not % returns of stocks alone.

In [ ]:
# Testing the output
sql = f"SELECT * FROM `{PROJECT_ID}.fin_news.prices`"

# submit & materialize via Storage API
job = bigquery_client.query(sql)
df_prices_raw = job.to_dataframe(bqstorage_client=bqstorage)

In [ ]:
df_prices_raw.head()

## Data - Daily Article Summaries

In [ ]:
# Testing the output
sql = f"SELECT * FROM `{PROJECT_ID}.fin_news.articles-daily-processed`"

# submit & materialize via Storage API
job = bigquery_client.query(sql)
df_articles_raw = job.to_dataframe(bqstorage_client=bqstorage)

df_articles_raw = df_articles_raw.rename(columns={'symbol': 'ticker'})

In [ ]:
df_articles_raw.groupby("ticker")['date'].agg(['min', 'max'])

# Data Processing

In [ ]:
ticker_list = df_articles_raw['ticker'].unique().tolist()

## Returns

- You decide to trade on day $t$
- You buy at Open of day $t + 1$
- You sell at Close of day $t + \text{holding period}$

In [ ]:
# Extending the time axis
def extend_time_axis(df):    
    all_dates = pd.date_range(start=df['date'].min(), end=df['date'].max())
    
    df_out = df.copy()
    df_out = df_out.set_index('date')
    df_out = df_out.reindex(all_dates)
    df_out = df_out.sort_index()
    df_out = df_out.bfill()
    df_out = df_out.shift(-1)
    df_out['ret'] = df_out['close'].shift(-pd.Timedelta(holding_period) // pd.Timedelta('1D')) - df_out["open"]
    df_out['pct_ret'] = df_out['ret'] / df_out["open"]

    return df_out

In [ ]:
df_prices = df_prices_raw.loc[
    (df_prices_raw['ticker'].isin(ticker_list))
].reset_index(drop=True)

In [ ]:
df_prices = df_prices.groupby(
    'ticker',
    as_index=True
).apply(
    extend_time_axis,
    include_groups=False
)

df_prices.index.set_names(['ticker', 'date'], inplace=True)
df_prices = df_prices.reset_index()
df_prices = df_prices.loc[
    df_prices['date'] >= train_start
]


df_prices = df_prices.dropna()
df_prices = df_prices.reset_index(drop=True)

## "Train" Dataset

For each ticker, we provide the news history (last 60 days) and target.

In [ ]:
df_articles = df_articles_raw.loc[
    df_articles_raw['date'].between(train_start, train_end)
].copy()

df_articles = df_articles.reset_index(drop=True)
df_articles['date'] = df_articles['date'].dt.tz_localize(None)

In [ ]:
df_time_ranges = df_articles.groupby(
    'ticker', 
    as_index=False
)['date'].agg(['min', 'max'])

df_time_ranges['min'] = df_time_ranges['min'] + pd.Timedelta(context_lookback)

df_time_ranges = df_time_ranges.groupby(
    'ticker',
).apply(
    lambda x: pd.DataFrame(
        pd.date_range(x['min'].values[0], x['max'].values[0], freq=train_stride),
        columns=['end_date']
    )
).reset_index(level=-1, drop=True).reset_index()

df_time_ranges['start_date'] = df_time_ranges['end_date'] - pd.Timedelta(train_stride)

In [ ]:
df_ret_train = df_prices.set_index(['ticker', 'date']).loc[
    df_time_ranges.set_index(['ticker', 'end_date']).index,
    ['pct_ret']
]

In [ ]:
df_ret_train.groupby('ticker').describe()

In [ ]:
df_ret_train.groupby('ticker').apply(
    lambda x: x.quantile([0.01, 0.05, 0.1, 0.95, 0.99]).T
)

In [ ]:
ds_train = [
    {
        "ticker": cur_row['ticker'],
        "start_date": cur_row['start_date'],
        "end_date": cur_row['end_date'],
        
        "articles": df_articles.loc[
            (df_articles['ticker'] == cur_row['ticker'])
            & (df_articles['date'].between(cur_row['start_date'], cur_row['end_date'])),
            ['date', 'texts']
        ].copy(),
        
        "return": df_prices.loc[
            (df_prices['ticker'] == cur_row['ticker'])
            & (df_prices['date'] == cur_row['end_date']),
            'pct_ret'
        ].values[0],
        
        "predictions_direction": [],
        "predictions_magnitude": [],
        "reflections": [] # a fancy term for experience
    }
    for _, cur_row in df_time_ranges.iterrows()
]

# Docstore setup

In [ ]:
tst = DocstoreExplorer(Wikipedia())

In [ ]:
tst.search('London Bridge')

In [ ]:
docstore = InMemoryDocstore()
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [ ]:
docstore.add(
    texts={
        str(uuid4()): x.page_content
        for x in documents
    }
)

In [ ]:
doc_expl = DocstoreExplorer(docstore)

In [ ]:
docstore.document

In [ ]:
docstore

In [ ]:
doc_expl.search('chocolate')

In [ ]:
generate_config = GenerationConfig(
    temperature=0,
    top_k=1,
    max_output_tokens=400
)

In [ ]:
# init picks up ADC

# get a Gemini model object
model = GenerativeModel(
    model_name=model_id,
    generation_config=generate_config
)

response = model.generate_content("How are you doing today?")

In [ ]:
chat = model.start_chat()

_ = chat.send_message("Can I ask you to remember a number for me? It's 10")
chat.send_message("What is the number I asked you to remember?")